In [ ]:
import sys
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ── paths (mesmo que o starter pack) ──────────────────────────────
PROJECT_ROOT = Path('..')
SCRIPTS_DIR  = PROJECT_ROOT / 'scripts'
KAGGLE_ROOT  = PROJECT_ROOT / 'ArtBench-10'
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

# ── reproducibilidade ──────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── device ────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('Device:', device)

In [ ]:
import csv
from artbench_local_dataset import load_kaggle_artbench10_splits

# carrega dataset completo
hf_ds    = load_kaggle_artbench10_splits(KAGGLE_ROOT)
train_hf = hf_ds["train"]
test_hf  = hf_ds["test"]

class_names = list(train_hf.features["label"].names)
print("Classes:", class_names)

# transform: [0, 1]
IMAGE_SIZE = 32
BATCH_SIZE = 64

transform = T.Compose([
    T.Resize(IMAGE_SIZE),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
])

class HFDatasetTorch(Dataset):
    def __init__(self, hf_split, transform=None, indices=None):
        self.ds = hf_split
        self.transform = transform
        self.indices = list(range(len(hf_split))) if indices is None else list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        ex = self.ds[self.indices[idx]]
        img = ex["image"]
        y   = int(ex["label"])
        x   = self.transform(img) if self.transform else img
        return x, y

# ── subset 20% (para desenvolvimento) ─────────────────────────────
def load_csv_ids(csv_path, col='train_id_original'):
    ids = []
    with open(csv_path, 'r') as f:
        for row in csv.DictReader(f):
            ids.append(int(row[col]))
    return ids

#train_ids = load_csv_ids(Path('training_20_percent.csv'))
#train_ds  = HFDatasetTorch(train_hf, transform=transform, indices=train_ids)
train_ds = HFDatasetTorch(train_hf, transform=transform)   # 50K (100%)
test_ds  = HFDatasetTorch(test_hf,  transform=transform)
print(f"Train: {len(train_ds)} | Test: {len(test_ds)}") 
#test_ds   = HFDatasetTorch(test_hf,  transform=transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

print(f"Train: {len(train_ds)} imagens | Test: {len(test_ds)} imagens")

In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.latent_dim = latent_dim

        # Encoder  3×32×32 → latent
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64,  4, stride=2, padding=1),  # →16×16
            nn.BatchNorm2d(64),  nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), # →8×8
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),# →4×4
            nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
        )
        self.fc_mu     = nn.Linear(256*4*4, latent_dim)
        self.fc_logvar = nn.Linear(256*4*4, latent_dim)

        # Decoder  latent → 3×32×32
        self.dec_fc = nn.Linear(latent_dim, 256*4*4)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1), # →8×8
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),  # →16×16
            nn.BatchNorm2d(64),  nn.ReLU(),
            nn.ConvTranspose2d(64, 3, 4, stride=2, padding=1),    # →32×32
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x).view(x.size(0), -1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z):
        h = self.dec_fc(z).view(-1, 256, 4, 4)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z    = self.reparameterize(mu, logvar)
        xhat = self.decode(z)
        return xhat, mu, logvar

# teste rápido de shapes
model = ConvVAE(latent_dim=128).to(device)
dummy = torch.randn(4, 3, 32, 32).to(device)
out, mu, logvar = model(dummy)
print("Input:", dummy.shape, "→ Output:", out.shape, "| mu:", mu.shape)

In [ ]:
def vae_loss(xhat, x, mu, logvar, beta=1.0):
    recon = F.binary_cross_entropy(xhat, x, reduction='sum') / x.size(0)
    kl    = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return recon + beta * kl, recon, kl


def train_vae(model, loader, epochs=20, lr=1e-3, beta=1.0):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history   = []
    model.train()

    for ep in range(epochs):
        total, r_total, kl_total = 0., 0., 0.
        for x, _ in tqdm(loader, desc=f"Epoch {ep+1}/{epochs}", leave=False):
            x = x.to(device)
            optimizer.zero_grad()
            xhat, mu, logvar = model(x)
            loss, recon, kl  = vae_loss(xhat, x, mu, logvar, beta=beta)
            loss.backward()
            optimizer.step()
            total    += loss.item()  * x.size(0)
            r_total  += recon.item() * x.size(0)
            kl_total += kl.item()   * x.size(0)

        n = len(loader.dataset)
        h = {'loss': total/n, 'recon': r_total/n, 'kl': kl_total/n}
        history.append(h)
        print(f"Epoch {ep+1:02d} | loss={h['loss']:.2f}  recon={h['recon']:.2f}  kl={h['kl']:.2f}")

    return history

In [ ]:
EPOCHS   = 100
BETA     = 0.5
LATENT   = 128

model   = ConvVAE(latent_dim=LATENT).to(device)
history = train_vae(model, train_loader, epochs=EPOCHS, lr=1e-3, beta=BETA)

# guardar modelo
Path('artifacts').mkdir(exist_ok=True)
torch.save(model.state_dict(), Path('artifacts') / 'vae_artbench_full.pt')
print("Modelo guardado!")

In [ ]:
def show_reconstructions(model, loader, n=8):
    model.eval()
    x, _ = next(iter(loader))
    x = x[:n].to(device)
    with torch.no_grad():
        xhat, _, _ = model(x)
    imgs = torch.cat([x.cpu(), xhat.cpu()], dim=0)
    grid = make_grid(imgs, nrow=n)
    plt.figure(figsize=(16, 4))
    plt.imshow(grid.permute(1,2,0).numpy())
    plt.title("Cima: originais | Baixo: reconstruções")
    plt.axis('off'); plt.tight_layout(); plt.show()

def show_samples(model, n=16):
    model.eval()
    with torch.no_grad():
        z    = torch.randn(n, model.latent_dim).to(device)
        imgs = model.decode(z).cpu()
    grid = make_grid(imgs, nrow=int(n**0.5))
    plt.figure(figsize=(6,6))
    plt.imshow(grid.permute(1,2,0).numpy())
    plt.title("Amostras geradas pelo VAE")
    plt.axis('off'); plt.tight_layout(); plt.show()

def plot_history(history):
    epochs = range(1, len(history)+1)
    plt.figure(figsize=(10,3))
    for i, key in enumerate(['loss','recon','kl']):
        plt.subplot(1,3,i+1)
        plt.plot(epochs, [h[key] for h in history])
        plt.title(key); plt.xlabel('Epoch')
    plt.tight_layout(); plt.show()

plot_history(history)
show_reconstructions(model, test_loader)
show_samples(model, n=16)